# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata properties (do not use dict access)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Authors (@id): {[author['@id'] for author in dataset.metadata.author]}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Spatial coverage: {dataset.metadata.spatialCoverage}")
print(f"Temporal coverage: {dataset.metadata.temporalCoverage}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This section inspects available record sets, fields, and columns, referencing everything by their `@id` as defined by the Croissant schema. The `mlcroissant` API allows exploration via record set listings.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print("Available RecordSets and their @id:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, list its fields (by @id)
for rs in record_sets:
    print(f"\nRecordSet '@id': {rs['@id']}")
    print("Fields and their @id:")
    if 'field' in rs:
        for field in rs['field']:
            print(f"    - @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
    else:
        print("    No fields listed.")

# Choose a record set for demonstration
if record_sets:
    main_record_set_id = record_sets[0]['@id']  # Use the first RecordSet's @id for future cells
    print(f"\nWill use RecordSet '@id': {main_record_set_id} for extraction and EDA.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

Note: All entities are referenced by `@id` for consistency, as required by the Croissant schema.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns for the chosen record set
print(f"Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, grouping, etc.

Reference all fields and columns by their `@id` for operations, as per guidelines.

In [ ]:
# Choose a numeric field @id for demonstration -- show available numeric fields
numeric_fields = []
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        if 'field' in rs:
            for field in rs['field']:
                # Heuristic: look for Float or Integer types
                dtype = field.get('dataType', '')
                if 'Float' in dtype or 'Integer' in dtype or 'Number' in dtype:
                    numeric_fields.append(field['@id'])
print(f"Numeric fields in {main_record_set_id}: {numeric_fields}")

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use the first found numeric field
else:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes(include=['number']).columns[0]

threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
# Find a field with type Text or categorical, not numeric
group_fields = []
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        if 'field' in rs:
            for field in rs['field']:
                dtype = field.get('dataType', '')
                if ('Text' in dtype or 'String' in dtype) and field['@id'] != numeric_field_id:
                    group_fields.append(field['@id'])
print(f"Grouping fields in {main_record_set_id}: {group_fields}")
if group_fields:
    group_field_id = group_fields[0]
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot a histogram of the selected numeric field and, if available, a barplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f'Histogram of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Barplot by group, if grouped_df exists
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    grouped_df[numeric_field_id].plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings from your exploration.

- Loaded dataset metadata and previewed key descriptive fields.
- Enumerated available record sets and referenced them by their `@id`.
- Extracted tabular records, selected numeric/categorical fields for EDA, filtered, normalized, and grouped data.
- Visualized distributions and relationships with matplotlib.

This workflow demonstrates reproducible FAIR data exploration referencing schema entities by their unique `@id`, leveraging `mlcroissant` for robust and transparent analysis.